## The game rules
- If total of either player or banker is 8 or 9, the round ends and return output as required format.
- Otherwise, two cases to consider, player stands or player draw another card from the shoe.
- if player stands i.e player total is 6 or 7, this scenario has two further cases.
    - case 1: banker total is 6 or 7 then banker stands also and output result.
    - case 2: banker total is 5 or less then banker draws a card then output result.
- if player drew third card (C3), this scenario has 6 further cases.
    -  case 1: banker total is 0,1,2 then banker draws a card and output.
    - case 2: banker total is 3 then banker draws a card if C3 is not 8 and output.
    - case 3: banker total is 4 then banker draws a card if C3 is between 2 and 7 and output.
    - case 4: banker total is 5 then banker draws a card if C3 is between 4 and 7 and output.
    - case 5: banker total is 6 then banker draws a card if C3 is between 6 and 7 and output.
    - case 6: banker total is 7 then banker stands and output.

## The Game Architecture 
- Define a function `create_deck` that will return 416 cards.
- Define a function `prepare_shoe` that will shuffle these 416 cards and use the first card burn rule.
- Assign the output from `prepare_shoe` to a variable so the shoe is not reshuffled in each round.
- Define a function `hand_total` that will calculate the totals of the player and banker in each round.
- Define a function `longest_straight_outcome` that will calculate the longest straight outcome at the end of each round.
- Define a function `game_result` that will return the result of each round in required format.
- Define the main function `play_game` where the rules of the game will be checked and the output will be returned until the cards are finished. 

In [0]:
# Need random function to shuffle the cards
import random

def create_deck():
    deck = []
    for shoe in range(8):  # 8 decks
        for suit in range(4):
            # add aces with hand value 1
            deck.append(("A", 1))
            # add cards 2 to 9 with corresponding hand value
            for i in range(2, 10):
                deck.append((str(i), i))
            # add 10, J, Q, K with hand value 0
            deck.append(("10", 0))
            deck.append(("J", 0))
            deck.append(("Q", 0))
            deck.append(("K", 0))
    return deck


# Prepare the shoe
def prepare_shoe():
    deck = create_deck()
    random.shuffle(deck)

    # First card burn rule
    cut_card = deck.pop(0)

    if cut_card[1] == 0:
        burn = 10
    else:
        burn = cut_card[1]
    deck = deck[burn:]
    return deck

In [0]:
# By definition hand value is the total value of cards modulo ten
def hand_total(hand):
    return sum(card[1] for card in hand) % 10

In [0]:

def longest_straight_outcome(player, banker, min_length=2):
    # Convert ranks to numeric values
    rank_map = {'A': 1,'2': 2,'3': 3,'4': 4,'5': 5,'6': 6,'7': 7,'8': 8,'9': 9,'10': 10,'J': 11,'Q': 12, 'K': 13
    }
    values = [rank_map[card[0]] for card in player + banker]
    values = sorted(set(values))
    longest = 1 
    current = 1
    for i in range(1, len(values)):
        if values[i] == values[i - 1] + 1: #check if the current value is one more than the previous value
            current += 1
            longest = max(longest, current)
        else:
            current = 1

    return longest

In [0]:
# required format output, to call this function when returning the results
def game_result(player, banker):
    player_cards = ",".join(card[0] for card in player)
    banker_cards = ",".join(card[0] for card in banker)
    Phand = hand_total(player)
    Bhand = hand_total(banker)

    if Phand > Bhand:
        outcome = "PLAYER"
    elif Bhand > Phand:
        outcome = "BANKER"
    else:
        outcome = "TIE"

    straight = longest_straight_outcome(player, banker)
    
    return (f"PHand - BHand - Outcome - Longest Straight Outcome" '\n' 
            f"{player_cards}- {banker_cards}-{outcome} -{straight}")

In [0]:
# assign prepared shoe on this cell to avoid reshuffling the shoe when the play_game function reruns
deck = prepare_shoe()

In [0]:
# game function, rerun only this cell until shoe is empty
def play_game():
    # Initial deal alternatively
    player = [deck.pop(0)]
    banker = [deck.pop(0)]
    player.append(deck.pop(0))
    banker.append(deck.pop(0))

    Phand = hand_total(player)
    Bhand = hand_total(banker)

    # Natural 8 or 9
    if Phand in [8, 9] or Bhand in [8, 9]:
        return game_result(player, banker)

    # Player stands on 6 or 7
    if Phand in [6, 7]:
        if Bhand <= 5:
            banker.append(deck.pop(0))
        return game_result(player, banker) # case Bhand in (6,7) is covered here

    # Player draws
    player_third = deck.pop(0) # assign for use below otherwise deck.pop(0) will pop a new card if called again
    player.append(player_third)

    third_value = player_third[1]

    # apply 6 banker rules
    if Bhand <= 2:
        banker.append(deck.pop(0))

    elif Bhand == 3:
        if third_value != 8:
            banker.append(deck.pop(0))

    elif Bhand == 4:
        if 2 <= third_value <= 7:
            banker.append(deck.pop(0))

    elif Bhand == 5:
        if third_value in [4, 5, 6, 7]:
            banker.append(deck.pop(0))

    elif Bhand == 6:
        if third_value in [6, 7]:
            banker.append(deck.pop(0))

    return game_result(player, banker) # case Bhand == 7 is covered here
print(play_game())

PHand - BHand - Outcome - Longest Straight Outcome
8,9- 5,5,9-BANKER -2
